In [ ]:
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
import numpy as np
from collections import Counter
from sklearn.model_selection import GridSearchCV
import warnings
from sklearn.tree import DecisionTreeClassifier, export_text
from xgboost import XGBClassifier
import re
import statsmodels.api as sm

In [ ]:
def analyze_logistic_regression(X_train, y_train, X_test, y_test, params, model_name="Logistic Regression"):
    # Train the scikit-learn model on the training set
    model_sk = LogisticRegression(**params)
    model_sk.fit(X_train, y_train)
    
    # Predictions (class labels) on training and test sets
    y_train_pred = model_sk.predict(X_train)
    y_test_pred  = model_sk.predict(X_test)

    # Predictions (probabilities) for AUC
    y_train_prob = model_sk.predict_proba(X_train)[:, 1]
    y_test_prob  = model_sk.predict_proba(X_test)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_prob)
    test_auc  = roc_auc_score(y_test,  y_test_prob)

    print(f"--- {model_name} ---")
    print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
    print("Training AUC:", train_auc)
    print("Test AUC:", test_auc)
    print("Test Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    
    # Refit with statsmodels on the training data (with a constant) for interpretability
    X_train_const = sm.add_constant(X_train)
    try:
        model_sm = sm.Logit(y_train, X_train_const).fit(disp=0)
    except np.linalg.LinAlgError:
        print("Warning: Singular matrix encountered; using regularized fit.")
        model_sm = sm.Logit(y_train, X_train_const).fit_regularized(method='l1', alpha=1e-4)
    print(model_sm.summary())
    
    # Display coefficients sorted by absolute effect
    coefs = model_sm.params.drop('const', errors='ignore')
    sorted_coefs = coefs.reindex(coefs.abs().sort_values(ascending=False).index)
    print("\nCoefficients (sorted by absolute effect):")
    print(sorted_coefs)
    
    return model_sk, model_sm  # Optionally return AUC values if you want to store them

In [ ]:
def analyze_random_forest(X_train, y_train, X_test, y_test, params, model_name="Random Forest"):
    model = RandomForestClassifier(**params)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    # Probability predictions for AUC
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob  = model.predict_proba(X_test)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_prob)
    test_auc  = roc_auc_score(y_test,  y_test_prob)

    print(f"--- {model_name} ---")
    print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
    print("Training AUC:", train_auc)
    print("Test AUC:", test_auc)
    print("Test Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    
    feat_imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    print("\nFeature Importances:")
    print(feat_imp)
    
    return model  # Optionally return AUC values if you want to store them

In [ ]:
def analyze_cart(X_train, y_train, X_test, y_test, params, model_name="CART"):
    model = DecisionTreeClassifier(**params)
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    # Probability predictions for AUC
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob  = model.predict_proba(X_test)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_prob)
    test_auc  = roc_auc_score(y_test,  y_test_prob)

    print(f"--- {model_name} ---")
    print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
    print("Training AUC:", train_auc)
    print("Test AUC:", test_auc)
    print("Test Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    
    feat_imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    print("\nFeature Importances:")
    print(feat_imp)
    
    tree_rules = export_text(model, feature_names=list(X_train.columns))
    print("\nDecision Tree Rules:\n", tree_rules)
    
    return model  # Optionally return AUC values if you want to store them

In [ ]:
def analyze_xgboost(X_train, y_train, X_test, y_test, params, model_name="XGBoost"):
    # Note: pass eval_metric='logloss' or 'auc' inside params or use it directly here
    model = XGBClassifier(**params, use_label_encoder=False, eval_metric='logloss')
    model.fit(X_train, y_train)
    
    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    # Probability predictions for AUC
    y_train_prob = model.predict_proba(X_train)[:, 1]
    y_test_prob  = model.predict_proba(X_test)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_prob)
    test_auc  = roc_auc_score(y_test,  y_test_prob)

    print(f"--- {model_name} ---")
    print("Training Accuracy:", accuracy_score(y_train, y_train_pred))
    print("Test Accuracy:", accuracy_score(y_test, y_test_pred))
    print("Training AUC:", train_auc)
    print("Test AUC:", test_auc)
    print("Test Confusion Matrix:\n", confusion_matrix(y_test, y_test_pred))
    
    feat_imp = pd.Series(model.feature_importances_, index=X_train.columns).sort_values(ascending=False)
    print("\nFeature Importances:")
    print(feat_imp)
    
    return model  # Optionally return AUC values if you want to store them

In [ ]:
def prepare_data_additive(df, top_list, pheno, top_n=10):
    """
    Prepare data for additive models without appending '_HET' to the top variants.
    """
    df = df.dropna(subset=[pheno])
    # Candidate features are taken directly from the list (no '_HET' appended)
    candidate_features = top_list.iloc[:, 0].tolist()
    selected_features = select_uncorrelated_features(df, candidate_features, threshold=0.95, top_n=top_n)
    selected_features.extend(covariates)
    print(f"Selected predictors for {pheno} (Additive):", selected_features)
    X = df[selected_features]
    y = df[pheno]
    return train_test_split(X, y, test_size=0.3, random_state=42)

In [ ]:
def prepare_data_het(df, top_list, pheno, top_n=10):
    """
    Prepare data for het models by appending '_HET' to the top variants.
    """
    df = df.dropna(subset=[pheno])
    candidate_features = [str(x) + '_HET' for x in top_list.iloc[:, 0].tolist()]
    selected_features = select_uncorrelated_features(df, candidate_features, threshold=0.95, top_n=top_n)
    selected_features.extend(covariates)
    print(f"Selected predictors for {pheno} (Het):", selected_features)
    X = df[selected_features]
    y = df[pheno]
    return train_test_split(X, y, test_size=0.3, random_state=42)


In [ ]:
def get_top_results(results_df):
    """
    Given a GridSearchCV results DataFrame, return all rows with the maximum mean_test_score.
    """
    max_score = results_df['mean_test_score'].max()
    top_results = results_df[results_df['mean_test_score'] == max_score]
    return top_results

In [ ]:
#%% Define function to select top uncorrelated features
def select_uncorrelated_features(X, candidate_features, threshold=0.95, top_n=10):
    """
    Select up to top_n features from candidate_features such that none is highly correlated
    (absolute correlation >= threshold) with any already selected feature.
    
    Parameters:
      - X: DataFrame containing candidate features.
      - candidate_features: List of candidate feature names (in priority order).
      - threshold: Correlation threshold above which features are considered redundant.
      - top_n: Maximum number of features to select.
    
    Returns: a list of selected feature names.
    """
    selected = []
    for feature in candidate_features:
        if feature not in X.columns:
            continue  # Skip if feature not in dataframe
        # If no features selected yet, add the first one.
        if len(selected) == 0:
            selected.append(feature)
        else:
            is_independent = True
            for sel in selected:
                corr_val = np.abs(X[[feature, sel]].corr().iloc[0, 1])
                if corr_val >= threshold:
                    is_independent = False
                    break
            if is_independent:
                selected.append(feature)
        if len(selected) >= top_n:
            break
    return selected


In [ ]:
def hyperparameter_logistic_regression(X_train, y_train):
    """
    Evaluate model hyperparameterization

    Parameters:
    - X_train, y_train: Training data and labels

    Returns:
    - Nothing
    - Will print information on best parameters and accuracy
    """
    # Create parameter grid manually to avoid conflicts
    param_grid_logreg = { 'penalty': ['l1', 'l2'], 'C': [1, 2, 5, 10], 'max_iter': [250, 500], 'solver': ['liblinear'] }
    
    # Create model
    logreg = LogisticRegression()
    
    # Run GridSearchCV
    grid_search_logreg = GridSearchCV(logreg, param_grid_logreg, cv=5, scoring='roc_auc', verbose=2, n_jobs=-1, return_train_score=True)
    
    # Fit to data
    grid_search_logreg.fit(X_train, y_train)

    #Save results into DF
    grid_search_results = get_top_results(pd.DataFrame(grid_search_logreg.cv_results_).sort_values(by="mean_test_score", ascending=False))
    
    # Best parameters & score
    return grid_search_results, grid_search_logreg.best_params_, grid_search_logreg.best_score_

In [ ]:
def hyperparameter_random_forest(X_train, y_train):
    """
    Evaluate random forest model hyperparameterization

    Parameters:
    - X_train, y_train: Training data and labels

    Returns:
    - rf_model_results
    - grid_search_rf.best_params_,
    - grid_search_rf.best_score_
    """
    # Create parameter grid for RandomForest
    param_grid_rf =  {
        'n_estimators': [50, 100, 150, 200],
        'max_depth': [None, 5, 7, 10, 15],
        'min_samples_split': [2, 3, 5, 7, 10],
        'max_features': ['sqrt', 'log2', 0.3, 0.5, None],
        'random_state': [42]
    }
    
    # Assuming you have X_train, y_train already defined
    
    # Initialize RandomForestClassifier
    rf = RandomForestClassifier()
    
    # GridSearchCV for RandomForest
    grid_search_rf = GridSearchCV(estimator=rf, param_grid=param_grid_rf, cv=5, n_jobs=-1, verbose=2, return_train_score=True,)
    
    # Fit the model
    grid_search_rf.fit(X_train, y_train)
    
    # Get the best model
    rf_model_results = get_top_results(pd.DataFrame(grid_search_rf.cv_results_).sort_values(by="mean_test_score", ascending=False))
    
    return rf_model_results, grid_search_rf.best_params_, grid_search_rf.best_score_

In [ ]:
def hyperparameter_cart(X, y):
    """
    Perform GridSearchCV to tune a DecisionTreeClassifier (CART).
    
    Parameters:
      - X: Predictor features
      - y: Target labels
    
    Returns:
      - results_df: DataFrame of grid search results sorted by mean_test_score (descending)
      - best_params_: Best hyperparameters found
      - best_score_: Best ROC AUC score from cross-validation
    """
    # Define parameter grid for CART
    param_grid_cart = {
        'criterion': ['gini', 'entropy'],
        'max_depth': [None, 3, 5, 7, 10],
        'min_samples_split': [2, 3, 5, 7, 10],
        'min_samples_leaf': [1, 2, 3, 5]
    }


    # Initialize a DecisionTreeClassifier
    model = DecisionTreeClassifier(random_state=42)
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(model, param_grid_cart, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1, return_train_score=True)
    grid_search.fit(X, y)
    
    # Convert results to a DataFrame and sort by mean_test_score descending
    results_df = pd.DataFrame(grid_search.cv_results_).sort_values(by="mean_test_score", ascending=False)
    
    return results_df, grid_search.best_params_, grid_search.best_score_


In [ ]:
def hyperparameter_xgboost(X, y):
    """
    Perform GridSearchCV to tune an XGBClassifier.
    
    Parameters:
      - X: Predictor features
      - y: Target labels
    
    Returns:
      - results_df: DataFrame of grid search results sorted by mean_test_score (descending)
      - best_params_: Best hyperparameters found
      - best_score_: Best ROC AUC score from cross-validation
    """
    # Define parameter grid for XGBoost
    param_grid_xgb = {
        'n_estimators': [50, 100, 150, 200],
        'max_depth': [3, 6, 8, 10, 12],
        'learning_rate': [0.01, 0.05, 0.075, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0]
    }
    # Initialize XGBClassifier
    model = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
    
    # Set up GridSearchCV
    grid_search = GridSearchCV(model, param_grid_xgb, cv=5, scoring='roc_auc', n_jobs=-1, verbose=1, return_train_score=True)
    grid_search.fit(X, y)
    
    # Convert results to DataFrame and sort by mean_test_score descending
    results_df = pd.DataFrame(grid_search.cv_results_).sort_values(by="mean_test_score", ascending=False)
    
    return results_df, grid_search.best_params_, grid_search.best_score_

In [ ]:
wk8remission = pd.read_csv('/home/eleiterw/Thesis/analysisFiles/wk8Remission.interesting.csv')
wk8response = pd.read_csv('/home/eleiterw/Thesis/analysisFiles/wk8Response.interesting.csv')
response = pd.read_csv('/home/eleiterw/Thesis/analysisFiles/response.interesting.csv')
remission = pd.read_csv('/home/eleiterw/Thesis/analysisFiles/remission.interesting.csv')
seasonalpattern = pd.read_csv('/home/eleiterw/Thesis/analysisFiles/seasonal.interesting.csv')
seasonalpattern['seasonalpattern_dropped'] = seasonalpattern['seasonalpattern'].replace(3, np.nan)  # Set unknowns as NaN
seasonalpattern = seasonalpattern.dropna(subset=['seasonalpattern_dropped'])
seasonalpattern['seasonalpattern'] = (seasonalpattern['seasonalpattern']-1).astype(int)
final_dfs = {'remission' : remission, 'response' : response, 'wk8remission' : wk8remission, 'wk8response' : wk8response, 'seasonalpattern' : seasonalpattern}

In [ ]:
wk8remissionTop = pd.read_csv('/scratch/eleiterw/AppliedProject/results/interestingWk8Remission.txt', header=None)
wk8responseTop = pd.read_csv('/scratch/eleiterw/AppliedProject/results/interestingWk8Response.txt', header=None)
remissionTop = pd.read_csv('/scratch/eleiterw/AppliedProject/results/interestingRemission.txt', header=None)
responseTop = pd.read_csv('/scratch/eleiterw/AppliedProject/results/interestingResponse.txt', header=None)
seasonalpatternTop = pd.read_csv('/scratch/eleiterw/AppliedProject/results/interestingSeasonal.txt', header=None)
top_df =  {'remission' : remissionTop, 'response' : responseTop, 'wk8remission' : wk8remissionTop, 'wk8response' : wk8responseTop, 'seasonalpattern' : seasonalpatternTop}

In [ ]:
print(responseTop.head())

In [ ]:
pattern_variant = re.compile(r'^\d+:\d+:[ACGT]+:[ACGT]+:rs\d+(_HET)?$')
phenotypes = ['remission', 'response', 'wk8remission', 'wk8response', 'seasonalpattern']
additive = {}
het = {}

# Loop through all dataframe columns
for pheno in phenotypes:
    df = final_dfs[pheno]
    # Prepare lists for the additive and het dataframe columns
    additive_cols = []
    het_cols = []
    for col in df.columns:
        if bool(pattern_variant.match(col)):
            # If it's a genetic variant column, choose based on whether it ends with _HET.
            if col.endswith('_HET'):
                het_cols.append(col)
            else:
                additive_cols.append(col)
        else:
            # For non-variant covariates, retain them in both dataframes.
            additive_cols.append(col)
            het_cols.append(col)
    
    # Create the new dataframes based on the column selections
    additive[pheno] = df[additive_cols]
    het[pheno] = df[het_cols]
    # Optional: Display the column names for confirmation
    print("Additive dataframe columns:", additive[pheno].columns.tolist())
    print("Het dataframe columns:", het[pheno].columns.tolist())

In [ ]:
covariates = ['race',
'gender',
'age',
'qidsctotal.b',
'yearsschool',
'currentempstat',
'seasonalpattern',
'psychotherapy',
'EVEC.1',
'EVEC.2',
'EVEC.3',
'EVEC.4',
'maritalstatus_M',
'maritalstatus_S',
'maritalstatus_W',
'maritalstatus_D',
'maritalstatus_X',
'maritalstatus_C',
'maritalstatus_P']

In [ ]:
results = {}
# Iterate over the phenotypes
for pheno in phenotypes:
    df = additive[pheno]
    df.dropna(subset=[pheno], inplace=True)

    if pheno == 'seasonalpattern':
        # You could define candidate features in a similar way if you have a top variant file for seasonalpattern
        candidate_features = top_df[pheno].iloc[:, 0].tolist()  # or modify as needed
        selected_features = select_uncorrelated_features(df, candidate_features, threshold=0.95, top_n=5)
        selected_features.extend(covariates)
        X = df[selected_features]
        X = X.drop(columns = ['seasonalpattern'])
        y = df['seasonalpattern_dropped'] - 1
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    else:
        X_train, X_test, y_train, y_test = prepare_data_additive(df, top_df[pheno], pheno, top_n=5)

    X_full = pd.concat([X_train, X_test])
    y_full = pd.concat([y_train, y_test])
    
    # Hyperparameter tuning for Logistic Regression
    results_log, best_param_log, best_score_log = hyperparameter_logistic_regression(X_full, y_full)
    results[f"{pheno}_log"] = (results_log, best_param_log, best_score_log)
    
    # Hyperparameter tuning for Random Forest
    results_rf, best_param_rf, best_score_rf = hyperparameter_random_forest(X_full, y_full)
    results[f"{pheno}_rf"] = (results_rf, best_param_rf, best_score_rf)
    
    # Hyperparameter tuning for CART (Decision Tree)
    results_cart, best_param_cart, best_score_cart = hyperparameter_cart(X_full, y_full)
    results[f"{pheno}_cart"] = (results_cart, best_param_cart, best_score_cart)
    
    # Hyperparameter tuning for XGBoost
    results_xgb, best_param_xgb, best_score_xgb = hyperparameter_xgboost(X_full, y_full)
    results[f"{pheno}_xgb"] = (results_xgb, best_param_xgb, best_score_xgb)
    
    # Print a summary for the current phenotype
    print(f"Finished tuning for {pheno}")
    print("Best Logistic Regression Params:", best_param_log)
    print("Best Random Forest Params:", best_param_rf)
    print("Best CART Params:", best_param_cart)
    print("Best XGBoost Params:", best_param_xgb)
    print("=" * 50)


In [ ]:
print(f"remission best parameters:\nscore: {results['remission_log'][2]}")
for i in results['remission_log'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_log'][2]}")
for i in results['response_log'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_log'][2]}")
for i in results['wk8remission_log'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_log'][2]}")
for i in results['wk8response_log'][0]['params']:
    print(i)

print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_log'][2]}")
for i in results['seasonalpattern_log'][0]['params']:
    print(i)

In [ ]:
print(f"remission best parameters:\nscore: {results['remission_rf'][2]}")
for i in results['remission_rf'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_rf'][2]}")
for i in results['response_rf'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_rf'][2]}")
for i in results['wk8remission_rf'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_rf'][2]}")
for i in results['wk8response_rf'][0]['params']:
    print(i)
    
print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_rf'][2]}")
for i in results['seasonalpattern_rf'][0]['params']:
    print(i)

In [ ]:
print(f"remission best parameters:\nscore: {results['remission_cart'][2]}")
for i in results['remission_cart'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_cart'][2]}")
for i in results['response_cart'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_cart'][2]}")
for i in results['wk8remission_cart'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_cart'][2]}")
for i in results['wk8response_cart'][0]['params']:
    print(i)

print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_cart'][2]}")
for i in results['seasonalpattern_cart'][0]['params']:
    print(i)

In [ ]:
print(f"remission best parameters:\nscore: {results['remission_xgb'][2]}")
for i in results['remission_xgb'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_xgb'][2]}")
for i in results['response_xgb'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_xgb'][2]}")
for i in results['wk8remission_xgb'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_xgb'][2]}")
for i in results['wk8response_xgb'][0]['params']:
    print(i)

print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_xgb'][2]}")
for i in results['seasonalpattern_xgb'][0]['params']:
    print(i)

In [ ]:
results = {}
# Iterate over the phenotypes
for pheno in phenotypes:
    df = het[pheno]
    df.dropna(subset=[pheno], inplace=True)
    
    if pheno == 'seasonalpattern':
        # For seasonalpattern, use a similar candidate approach as additive but with '_HET'
        candidate_features = [str(x) + '_HET' for x in top_df[pheno].iloc[:, 0].tolist()]
        selected_features = select_uncorrelated_features(df, candidate_features, threshold=0.95, top_n=5)
        selected_features.extend(covariates)
        X = df[selected_features]
        # You may want to drop the target column (or any other unneeded columns) as in your additive branch.
        X = X.drop(columns=['seasonalpattern'])
        y = df['seasonalpattern_dropped'] - 1
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    else:
        X_train, X_test, y_train, y_test = prepare_data_het(df, top_df[pheno], pheno, top_n=5)
    
    # Merge train and test sets for internal CV (if your hyperparameter routines use full data)
    X_full = pd.concat([X_train, X_test])
    y_full = pd.concat([y_train, y_test])

    # Hyperparameter tuning for Logistic Regression
    results_log, best_param_log, best_score_log = hyperparameter_logistic_regression(X_full, y_full)
    results[f"{pheno}_log"] = (results_log, best_param_log, best_score_log)
    
    # Hyperparameter tuning for Random Forest
    results_rf, best_param_rf, best_score_rf = hyperparameter_random_forest(X, y)
    results[f"{pheno}_rf"] = (results_rf, best_param_rf, best_score_rf)
    
    # Hyperparameter tuning for CART (Decision Tree)
    results_cart, best_param_cart, best_score_cart = hyperparameter_cart(X, y)
    results[f"{pheno}_cart"] = (results_cart, best_param_cart, best_score_cart)
    
    # Hyperparameter tuning for XGBoost
    results_xgb, best_param_xgb, best_score_xgb = hyperparameter_xgboost(X, y)
    results[f"{pheno}_xgb"] = (results_xgb, best_param_xgb, best_score_xgb)
    
    # Print a summary for the current phenotype
    print(f"Finished tuning for {pheno}")
    print("Best Logistic Regression Params:", best_param_log)
    print("Best Random Forest Params:", best_param_rf)
    print("Best CART Params:", best_param_cart)
    print("Best XGBoost Params:", best_param_xgb)
    print("=" * 50)


In [ ]:
print(f"remission best parameters:\nscore: {results['remission_log'][2]}")
for i in results['remission_log'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_log'][2]}")
for i in results['response_log'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_log'][2]}")
for i in results['wk8remission_log'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_log'][2]}")
for i in results['wk8response_log'][0]['params']:
    print(i)

print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_log'][2]}")
for i in results['seasonalpattern_log'][0]['params']:
    print(i)

In [ ]:
print(f"remission best parameters:\nscore: {results['remission_rf'][2]}")
for i in results['remission_rf'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_rf'][2]}")
for i in results['response_rf'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_rf'][2]}")
for i in results['wk8remission_rf'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_rf'][2]}")
for i in results['wk8response_rf'][0]['params']:
    print(i)
    
print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_rf'][2]}")
for i in results['seasonalpattern_rf'][0]['params']:
    print(i)

In [ ]:
print(f"remission best parameters:\nscore: {results['remission_cart'][2]}")
for i in results['remission_cart'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_cart'][2]}")
for i in results['response_cart'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_cart'][2]}")
for i in results['wk8remission_cart'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_cart'][2]}")
for i in results['wk8response_cart'][0]['params']:
    print(i)

print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_cart'][2]}")
for i in results['seasonalpattern_cart'][0]['params']:
    print(i)

In [ ]:
print(f"remission best parameters:\nscore: {results['remission_xgb'][2]}")
for i in results['remission_xgb'][0]['params']:
    print(i)
print(f"response best parameters:\nscore: {results['response_xgb'][2]}")
for i in results['response_xgb'][0]['params']:
    print(i)
    
print(f"wk8remission best parameters:\nscore: {results['wk8remission_xgb'][2]}")
for i in results['wk8remission_xgb'][0]['params']:
    print(i)

print(f"wk8response best parameters:\nscore: {results['wk8response_xgb'][2]}")
for i in results['wk8response_xgb'][0]['params']:
    print(i)

print(f"seasonal pattern best parameters:\nscore: {results['seasonalpattern_xgb'][2]}")
for i in results['seasonalpattern_xgb'][0]['params']:
    print(i)

In [ ]:
# Ideal Parameters for Additive Models (New Selection from Tuning Results)
ideal_params_add_logreg = {
    'remission': {'C': 2, 'max_iter': 500, 'penalty': 'l1', 'solver': 'liblinear'},       # Score ~0.6329
    'response': {'C': 10, 'max_iter': 250, 'penalty': 'l1', 'solver': 'liblinear'},        # Score ~0.6700
    'wk8remission': {'C': 2, 'max_iter': 250, 'penalty': 'l2', 'solver': 'liblinear'},     # Score ~0.7745 (chose lower max_iter option)
    'wk8response': {'C': 10, 'max_iter': 500, 'penalty': 'l1', 'solver': 'liblinear'},     # Score ~0.6304
    'seasonalpattern': {'C': 5, 'max_iter': 250, 'penalty': 'l1', 'solver': 'liblinear'},  # Score ~0.6870
}

ideal_params_add_rf = {
    'remission': {'max_depth': 7, 'max_features': 0.3, 'min_samples_split': 2, 'n_estimators': 100, 'random_state': 42},  # Score ~0.6105
    'response': {'max_depth': 10, 'max_features': 'sqrt', 'min_samples_split': 10, 'n_estimators': 100, 'random_state': 42},  # Score ~0.7035
    'wk8remission': {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 200, 'random_state': 42},  # Score ~0.7262
    'wk8response': {'max_depth': 5, 'max_features': 0.3, 'min_samples_split': 3, 'n_estimators': 100, 'random_state': 42},    # Score ~0.6934
    'seasonalpattern': {'max_depth': None, 'max_features': 0.5, 'min_samples_split': 2, 'n_estimators': 50, 'random_state': 42},# Score ~0.7085
}

ideal_params_add_cart = {
    'remission': {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2},      # Selected from top remission options (~0.5847)
    'response': {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2},    # One option for response (~0.6064)
    'wk8remission': {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10},   # Selected for wk8remission (~0.6606)
    'wk8response': {'criterion': 'gini', 'max_depth': 7, 'min_samples_leaf': 5, 'min_samples_split': 2},    # One option for wk8response (~0.5604)
    'seasonalpattern': {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 10},# Option for seasonal pattern (~0.6377)
}

ideal_params_add_xgb = {
    'remission': {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0},   # Score ~0.6366
    'response': {'colsample_bytree': 0.8, 'learning_rate': 0.075, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0},  # Chosen similar option for response
    'wk8remission': {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 150, 'subsample': 1.0},# Slightly higher n_estimators option for wk8remission
    'wk8response': {'colsample_bytree': 0.8, 'learning_rate': 0.075, 'max_depth': 3, 'n_estimators': 100, 'subsample': 1.0},# Chosen for wk8response
    'seasonalpattern': {'colsample_bytree': 0.8, 'learning_rate': 0.075, 'max_depth': 3, 'n_estimators': 50, 'subsample': 1.0}, # Option for seasonal pattern
}


# Ideal Parameters for Het Models (New Selection from Tuning Results)
ideal_params_het_logreg = {
    'remission': {'C': 5, 'max_iter': 250, 'penalty': 'l2', 'solver': 'liblinear'},
    'response': {'C': 2, 'max_iter': 500, 'penalty': 'l1', 'solver': 'liblinear'},
    'wk8remission': {'C': 2, 'max_iter': 250, 'penalty': 'l1', 'solver': 'liblinear'},
    'wk8response': {'C': 10, 'max_iter': 250, 'penalty': 'l1', 'solver': 'liblinear'},
    'seasonalpattern': {'C': 1, 'max_iter': 250, 'penalty': 'l1', 'solver': 'liblinear'},
}

ideal_params_het_rf = {
    'remission': {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 10, 'n_estimators': 150, 'random_state': 42},
    'response': {'max_depth': 7, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 100, 'random_state': 42},
    'wk8remission': {'max_depth': 5, 'max_features': 'log2', 'min_samples_split': 10, 'n_estimators': 150, 'random_state': 42},
    'wk8response': {'max_depth': 5, 'max_features': 'sqrt', 'min_samples_split': 5, 'n_estimators': 100, 'random_state': 42},
    'seasonalpattern': {'max_depth': 10, 'max_features': 'log2', 'min_samples_split': 10, 'n_estimators': 150, 'random_state': 42},
}

ideal_params_het_cart = {
    'remission': {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2},
    'response': {'criterion': 'entropy', 'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2},
    'wk8remission': {'criterion': 'gini', 'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 10},
    'wk8response': {'criterion': 'gini', 'max_depth': 7, 'min_samples_leaf': 5, 'min_samples_split': 3},
    'seasonalpattern': {'criterion': 'gini', 'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 10},
}

ideal_params_het_xgb = {
    'remission': {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100, 'subsample': 1.0},
    'response': {'colsample_bytree': 0.8, 'learning_rate': 0.075, 'max_depth': 6, 'n_estimators': 100, 'subsample': 1.0},
    'wk8remission': {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 6, 'n_estimators': 100, 'subsample': 0.8},
    'wk8response': {'colsample_bytree': 0.8, 'learning_rate': 0.05, 'max_depth': 10, 'n_estimators': 100, 'subsample': 0.8},
    'seasonalpattern': {'colsample_bytree': 0.8, 'learning_rate': 0.075, 'max_depth': 8, 'n_estimators': 150, 'subsample': 0.8},
}

In [ ]:
#%% Analyze Additive Models (RF, CART, and XGBoost)
print("========== ADDITIVE MODELS (No Logistic Regression) ==========")
for pheno in phenotypes:
    print(f"\n***** Phenotype: {pheno} *****")
    df = additive[pheno]
    if pheno == 'seasonalpattern':
        # You could define candidate features in a similar way if you have a top variant file for seasonalpattern
        candidate_features = top_df[pheno].iloc[:, 0].tolist()  # or modify as needed
        selected_features = select_uncorrelated_features(df, candidate_features, threshold=0.95, top_n=5)
        selected_features.extend(covariates)
        X = df[selected_features]
        X = X.drop(columns = ['seasonalpattern'])
        y = df['seasonalpattern_dropped'] - 1
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    else:
        X_train, X_test, y_train, y_test = prepare_data_additive(df, top_df[pheno], pheno, top_n=5)
    
    print("\n--- Additive Random Forest ---")
    params = ideal_params_add_rf[pheno]
    model_rf = analyze_random_forest(X_train, y_train, X_test, y_test, params, model_name=f"Additive RF {pheno}")
    
    print("\n--- Additive CART ---")
    params = ideal_params_add_cart[pheno]
    model_cart = analyze_cart(X_train, y_train, X_test, y_test, params, model_name=f"Additive CART {pheno}")
    
    print("\n--- Additive XGBoost ---")
    params = ideal_params_add_xgb[pheno]
    model_xgb = analyze_xgboost(X_train, y_train, X_test, y_test, params, model_name=f"Additive XGB {pheno}")
    
    print("="*50)

In [ ]:
#%% Analyze Het Models (RF, CART, and XGBoost)
print("\n========== HET MODELS (No Logistic Regression) ==========")
for pheno in phenotypes:
    print(f"\n***** Phenotype: {pheno} *****")
    df = het[pheno]
    if pheno == 'seasonalpattern':
        # For seasonalpattern, use a similar candidate approach as additive but with '_HET'
        candidate_features = [str(x) + '_HET' for x in top_df[pheno].iloc[:, 0].tolist()]
        selected_features = select_uncorrelated_features(df, candidate_features, threshold=0.95, top_n=5)
        selected_features.extend(covariates)
        X = df[selected_features]
        # You may want to drop the target column (or any other unneeded columns) as in your additive branch.
        X = X.drop(columns=['seasonalpattern'])
        y = df['seasonalpattern_dropped'] - 1
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    else:
        X_train, X_test, y_train, y_test = prepare_data_het(df, top_df[pheno], pheno, top_n=5)
    
    print("\n--- Het Random Forest ---")
    params = ideal_params_het_rf[pheno]
    model_rf = analyze_random_forest(X_train, y_train, X_test, y_test, params, model_name=f"Het RF {pheno}")
    
    print("\n--- Het CART ---")
    params = ideal_params_het_cart[pheno]
    model_cart = analyze_cart(X_train, y_train, X_test, y_test, params, model_name=f"Het CART {pheno}")
    
    print("\n--- Het XGBoost ---")
    params = ideal_params_het_xgb[pheno]
    model_xgb = analyze_xgboost(X_train, y_train, X_test, y_test, params, model_name=f"Het XGB {pheno}")
    
    print("="*50)

In [ ]:
# ### Initialize an empty dictionary to store results
# results = {}
# phenotypes = ['remission', 'response', 'wk8remission', 'wk8response', 'seasonalpattern']

# # Iterate over the phenotypes
# for pheno in phenotypes:
#     df = final_dfs[pheno]
#     df.dropna(subset=pheno, inplace = True)
    
#     # Define the predictors (X) and target (y)
#     X = df.drop(columns=['remission', 'response', 'wk8remission', 'wk8response','seasonalpattern', 'FID', 'FID_y', 'FID_x', 'IID', 'PHENOTYPE', 'PAT', 'MAT', 'SEX', 'qidscpctchange', 'consentMonth', 'consentYear', 'currentonsetMonth', 'currentonsetYear'])  # Drop phenotype column, 'FID', 'IID'
#     y = df[pheno]  # This is the phenotype we're predicting
    
#     # Split the data into training and testing sets
#     X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    
#     # Logistic Regression with Regularization
#     logreg_accuracy, logreg_auc, logreg_cm, logreg_model = train_and_evaluate_model(X_train, y_train, X_test, y_test, model_type="logreg", regularization='l2')

#     if pheno=='seasonalpattern':
#         X = df.drop(columns=['remission', 'response', 'wk8remission', 'wk8response', 'seasonalpattern', 'FID', 'FID_y', 'FID_x', 'IID', 'PHENOTYPE', 'PAT', 'MAT', 'SEX', 'qidscpctchange', 'consentMonth', 'consentYear', 'currentonsetMonth', 'currentonsetYear'])  # Drop phenotype column, 'FID', 'IID'
#         y = df['seasonalpattern_dropped']  # This is the phenotype we're predicting
#         X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#     # Random Forest
#     rf_accuracy, rf_auc, rf_cm, rf_model = train_and_evaluate_model(X_train, y_train, X_test, y_test, model_type="rf", max_depth=10)
    
#     # Store results for both models
#     results[pheno] = {
#         'logreg': {
#             'accuracy': logreg_accuracy,
#             'AUC': logreg_auc,
#             'confusion_matrix': logreg_cm
#         },
#         'random_forest': {
#             'accuracy': rf_accuracy,
#             'AUC': rf_auc,
#             'confusion_matrix': rf_cm
#         }
#     }

#     # Optionally, you can store the models as well, if you'd like to access them later
#     results[pheno]['logreg_model'] = logreg_model
#     results[pheno]['rf_model'] = rf_model
#     results[pheno]['columns'] = X.columns

# # Output the results
# for pheno, result in results.items():
#     print(f"Results for {pheno}:")
#     print("Logistic Regression:")
#     print(f"Accuracy: {result['logreg']['accuracy']:.4f}")
#     print(f"AUC: {result['logreg']['AUC']:.4f}")
#     print(f"Confusion Matrix:\n{result['logreg']['confusion_matrix']}")
#     print("\nRandom Forest:")
#     print(f"Accuracy: {result['random_forest']['accuracy']:.4f}")
#     print(f"AUC: {result['random_forest']['AUC']:.4f}")
#     print(f"Confusion Matrix:\n{result['random_forest']['confusion_matrix']}")
#     print("="*40)


In [ ]:
# feature_names = results['remission']['columns']
# coeff = results['remission']['logreg_model'].coef_[0]
# map = {'feature' : feature_names, 'coeff': coeff}
# map_df = pd.DataFrame(map).sort_values(by = 'coeff', ascending=False)